# Multi-region model: zero-shot transfer and fine-tuning (Sec. 3.3; Fig. 5, 6, A6–A9)

Trains one Transformer on all source regions together (CH, FR, NOR, ISL; the multi-region or "pooled" model), compares its zero-shot transfer to that of the single-region models of `2.3 Cross_region`, and fine-tunes it with 10% of the measurements of each target region.

| Paper figure | Content | Saved as (in `figures/paperTF/`) |
|---|---|---|
| Fig. 5 | Annual RMSE of zero-shot transfer: multi-region vs single-region models, per target region | `fig5_zs_pooled_vs_ss_barplot_a.png` |
| Fig. A6 | Same for winter RMSE | `appendix/A6_zs_pooled_vs_ss_barplot_w.png` |
| Fig. A7 | Modeled vs observed PMB of all zero-shot models per target region | `appendix/A7_zs_pooled_vs_ss_scatter_all.png` |
| Fig. 6 | Annual RMSE of the multi-region model without and with fine-tuning (head only, full) | `fig6_pooled_ft_gain_barplot.png` |
| Fig. A8 | Annual RMSE versus fine-tuning fraction | `appendix/A8_sensitivity_FT.png` |
| Fig. A9 | Training and validation loss during fine-tuning (10%) | `appendix/A9_FT_loss_curves.png` |

**Steps**
1. Load the stake datasets and monthly inputs, and split Central Asia, Alaska and Italy/Austria into sub-regions.
2. Build the training dataset of the multi-region model and the fine-tuning/test datasets of the target regions.
3. Load the multi-region model and the single-region models of `2.3 Cross_region`, and compare their zero-shot transfer (Fig. 5, A6, A7).
4. Fine-tune the multi-region model on each target region (Fig. 6).
5. Sensitivity of fine-tuning to the fraction of local data (Fig. A8, A9).

Datasets and models are cached in `cfg.dataPath/cache/TF_Europe/FD_MODEL/`. The single-region models are read from `cfg.dataPath/cache/TF_Europe/CROSS_REGION/models/`. The functions used here live in `scripts/dataset/pooled_experiment.py`, `scripts/models/pooled_model.py`, `scripts/models/pooled_figures.py` and `scripts/models/ft_sensitivity.py`.

## Setting up:

In [ ]:
import os

# Silence KeOps (used by geomloss for Sinkhorn distances) before it is imported
os.environ["PYKEOPS_VERBOSE"] = "0"
os.environ["KEOPS_VERBOSE"] = "0"

import torch

# Initialise CUDA before the other imports: changing this order can change the
# GPU results slightly (~1e-7)
if torch.cuda.is_available():
    torch.cuda.init()

import sys
import logging
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import massbalancemachine as mbm

sys.path.append(os.path.join(os.getcwd(), '../../'))  # Add root of repo to import MBM
# Shared config (paths, MONTHLY_COLS, STATIC_COLS, REGION_LABELS, ...), model settings
# (PARAMS_TRANSFORMER, PARAMS_FT, N_EPOCHS, MODEL_DATE) and helpers
from papers.Transformer_TF.scripts.config_TF_Europe import *
from papers.Transformer_TF.scripts.config_models import *
from papers.Transformer_TF.scripts.dataset import *
from papers.Transformer_TF.scripts.plotting import *
from papers.Transformer_TF.scripts.models import *

warnings.filterwarnings('ignore')
%load_ext autoreload
%autoreload 2

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')

# Call config file (data paths, seed, etc.)
cfg = mbm.EuropeTFConfig()
mbm.utils.seed_all(cfg.seed)  # reproducibility
mbm.utils.free_up_cuda()
mbm.plots.use_mbm_style()

# Use the GPU if one is visible
device = torch.device("cuda" if torch.cuda.device_count() > 0 else "cpu")
print(f"Device: {device}")

# Cache folder of the multi-region experiments (datasets, models)
BASE_DIR = Path(cfg.dataPath) / path_cache / "FD_MODEL"
CACHE_DIR = BASE_DIR / "datasets"
os.makedirs(CACHE_DIR, exist_ok=True)
print(f"Base directory for data: {BASE_DIR}")

os.makedirs("figures/paperTF/appendix", exist_ok=True)

## Data:
### Stakes and monthly inputs:
With `run_flag_by_code = False` (default), the monthly data are loaded from `WGMS/CrossRegional/monthly_cache/` instead of being recomputed. Central Asia and Alaska are split into their RGI O2 sub-regions, and Italy/Austria by country.

In [ ]:
SOURCE_REGIONS = ['CH', 'NOR', 'ISL', "FR"]  # training regions of the multi-region model
TARGET_REGIONS = ['IT_AT', 'SJM', 'CENTRALASIA', 'ALA']
# Target (sub-)regions evaluated in the paper
TARGET_REGIONS_SUB = ['IT', 'AT', 'SJM', 'ALA', 'CA_3', 'CA_4']

data = load_pooled_experiment_data(cfg, SOURCE_REGIONS, TARGET_REGIONS,
                                   TARGET_REGIONS_SUB)
dfs = data["dfs"]
monthly_cache = data["monthly_cache"]
res_xreg_by_source = data["res_xreg_by_source"]
months_head_pad = data["months_head_pad"]
months_tail_pad = data["months_tail_pad"]

# Number of stake glaciers per region (shown in the panel titles)
REGION_GLACIER_COUNTS = count_glaciers_per_region(dfs)
print(REGION_GLACIER_COUNTS)

### Model datasets:
**Multi-region training set:** all glaciers of the source regions (80/20 train/validation split).

**Fine-tuning / test sets:** in each target region, 10% of the measurement IDs (winter and annual drawn separately) are used for fine-tuning and the remaining 90% for testing. This is repeated `N_REPEATS` times with different seeds.

In [ ]:
assets_full = build_pooled_assets(
    res_xreg_by_source,
    SOURCE_REGIONS,
    monthly_cols=MONTHLY_COLS,
    static_cols=STATIC_COLS,
    cfg=cfg,
    months_head_pad=months_head_pad,
    months_tail_pad=months_tail_pad,
    cache_path=CACHE_DIR / "assets_foundation_100pct.joblib",
)

# FT_FRAC (fraction of local measurements used for fine-tuning) and the number of
# repeats are set in scripts/config_models.py
N_REPEATS = FT_N_REPEATS

ft_assets_id = build_ft_assets_id_split(
    monthly_cache,
    TARGET_REGIONS_SUB,
    monthly_cols=MONTHLY_COLS,
    static_cols=STATIC_COLS,
    cfg=cfg,
    cache_dir=CACHE_DIR,
    ft_frac=FT_FRAC,
    n_repeats=N_REPEATS,
)

## Zero-shot transfer:
### Models:
The multi-region model and the single-region models of `2.3 Cross_region` (one per source region). Each model standardises its inputs with the statistics of its own training data. With `TRAIN_FLAG = False`, missing models are not trained.

In [ ]:
TRAIN_FLAG = False

model_pooled, ds_pooled_scaler = load_pooled_model(
    cfg,
    device,
    assets_full,
    params=PARAMS_TRANSFORMER,
    models_dir=BASE_DIR / "models" / "foundation",
    epochs=N_EPOCHS,
    date=MODEL_DATE,
    train_flag=TRAIN_FLAG,
)

single_source_models, single_source_assets, single_source_scalers = load_single_source_models(
    cfg,
    device,
    res_xreg_by_source,
    SOURCE_REGIONS,
    monthly_cols=MONTHLY_COLS,
    static_cols=STATIC_COLS,
    months_head_pad=months_head_pad,
    months_tail_pad=months_tail_pad,
    params=PARAMS_TRANSFORMER,
    models_base=Path(cfg.dataPath) / path_cache / "CROSS_REGION" / "models",
    epochs=N_EPOCHS,
    date=MODEL_DATE,
    train_flag=TRAIN_FLAG,
)

### Metrics:
All models are evaluated on all stakes of each target region.

In [ ]:
# Full-region test set of each target region
eval_configs = build_zero_shot_test_sets(monthly_cache,
                                         TARGET_REGIONS_SUB,
                                         monthly_cols=MONTHLY_COLS,
                                         static_cols=STATIC_COLS,
                                         cfg=cfg,
                                         cache_dir=CACHE_DIR)

df_metrics_comparison, region_pred_target_ranges = zero_shot_comparison_metrics(
    cfg, device, eval_configs, model_pooled, ds_pooled_scaler,
    single_source_models, single_source_scalers)
print_metric_summary(df_metrics_comparison, region_pred_target_ranges)

### Fig. 5 and A6: RMSE per target region

In [ ]:
fig = plot_zs_pooled_vs_single_barplot(
    df_metrics_comparison,
    metric_key="RMSE_annual",
    metric_label="RMSE annual (m w.e.)",
    source_regions=SOURCE_REGIONS,
    glacier_counts=REGION_GLACIER_COUNTS,
    out_path="figures/paperTF/fig5_zs_pooled_vs_ss_barplot_a.png",
    suptitle="Multi-region versus single-region zero-shot transfer",
)

In [ ]:
fig = plot_zs_pooled_vs_single_barplot(
    df_metrics_comparison,
    metric_key="RMSE_winter",
    metric_label="RMSE winter (m w.e.)",
    source_regions=SOURCE_REGIONS,
    glacier_counts=REGION_GLACIER_COUNTS,
    out_path="figures/paperTF/appendix/A6_zs_pooled_vs_ss_barplot_w.png",
    suptitle="Multi-region versus single-region zero-shot transfer: winter mass balance",
    global_ylim=True,
)

### Fig. A7: modeled vs observed PMB

In [ ]:
fig = plot_zs_pooled_vs_single_scatter(
    cfg,
    device,
    df_metrics_comparison,
    eval_configs,
    SOURCE_REGIONS,
    model_pooled,
    ds_pooled_scaler,
    single_source_models,
    single_source_scalers,
    out_path="figures/paperTF/appendix/A7_zs_pooled_vs_ss_scatter_all.png",
)

## Fine-tuning:
Fine-tune the multi-region model on the 10% FT set of each target region and repeat: either only the output head (`head_only`) or all weights (`full`), with the settings in `PARAMS_FT`. Models are cached in `FD_MODEL/models/finetuned/` and only trained if missing.

In [ ]:
ft_models_id = finetune_id_split_models(
    cfg,
    device,
    model_pooled,
    ds_pooled_scaler,
    ft_assets_id,
    TARGET_REGIONS_SUB,
    models_dir_ft=BASE_DIR / "models" / "finetuned",
    params=PARAMS_TRANSFORMER,
    ft_params=PARAMS_FT,
    date=MODEL_DATE,
)

### Metrics:
The zero-shot (`no_ft`) and fine-tuned models are evaluated on the same 90% test set of each repeat; metrics are averaged over the repeats.

In [ ]:
df_ft_metrics_id, df_ft_metrics_id_raw = ft_metrics_id_split(
    cfg, device, model_pooled, ds_pooled_scaler, ft_assets_id, ft_models_id,
    TARGET_REGIONS_SUB)

### Fig. 6: fine-tuning gain

In [ ]:
fig = plot_ft_gain_barplot(df_ft_metrics_id,
                           glacier_counts=REGION_GLACIER_COUNTS,
                           out_path="figures/paperTF/fig6_pooled_ft_gain_barplot.png")

## Sensitivity to the fine-tuning fraction (Fig. A8, A9):
Fine-tuning fractions of 5–50% of the target-region measurement IDs, `N_REPEATS` random repeats each, head-only and full fine-tuning.

**Design**
- Each repeat uses the **same seed and ID shuffle as the main experiment** (`cfg.seed + rep`; winter and annual IDs shuffled separately). The 10% subset is therefore identical to the FT set of the main results.
- The **test set is fixed across fractions**: the IDs left after the largest (50%) FT subset. FT subsets are **nested** (5% ⊂ 10% ⊂ … ⊂ 50%), so differences between fractions come only from the amount of FT data. (Absolute RMSEs at 10% can differ slightly from Fig. 6, which evaluates on the remaining 90%.)
- Fine-tuning follows `finetune_transformer_on_target` (80/20 train/val split of the FT data, weighted sampler, AdamW with `lr × lr_factor`, ReduceLROnPlateau, early stopping, best-validation checkpoint) and also stores the per-epoch train/val loss (epoch 0 = multi-region model before fine-tuning).
- Runs whose FT set has fewer than `MIN_FT_SEQS` sequences are skipped (degenerate validation split) and show up as missing points.

Models and loss histories are cached in `FD_MODEL/ft_fraction_sensitivity/models/`, and the per-run metrics are saved to `SENS_RESULTS_CSV`.

In [ ]:
FT_FRACS_SENS = [0.05, 0.10, 0.15, 0.2, 0.25, 0.3, 0.50]
STRATEGIES_SENS = ["head_only", "full"]
MIN_FT_SEQS = 5  # skip runs with fewer FT sequences (degenerate 80/20 split)

SENS_DIR = BASE_DIR / "ft_fraction_sensitivity"
SENS_RESULTS_CSV = SENS_DIR / f"ft_fraction_sensitivity_{MODEL_DATE}.csv"

# Nested FT subsets and fixed test set per region and repeat
sens_assets, sens_test, df_sens_sizes = build_ft_fraction_splits(
    cfg,
    monthly_cache,
    TARGET_REGIONS_SUB,
    fracs=FT_FRACS_SENS,
    n_repeats=N_REPEATS,
    monthly_cols=MONTHLY_COLS,
    static_cols=STATIC_COLS,
    cache_dir=CACHE_DIR,
    ft_assets_id=ft_assets_id,  # check that the 10% subsets match the main FT sets
)

In [ ]:
# Fine-tune (or load) and evaluate one model per region, fraction, strategy and repeat
df_sens, sens_histories = run_ft_fraction_sensitivity(
    cfg,
    device,
    model_pooled,
    ds_pooled_scaler,
    sens_assets,
    sens_test,
    TARGET_REGIONS_SUB,
    fracs=FT_FRACS_SENS,
    strategies=STRATEGIES_SENS,
    n_repeats=N_REPEATS,
    models_dir=SENS_DIR / "models",
    params=PARAMS_TRANSFORMER,
    ft_params=PARAMS_FT,
    date=MODEL_DATE,
    min_ft_seqs=MIN_FT_SEQS,
    results_csv=SENS_RESULTS_CSV,
)

# RMSE per fraction, change vs zero-shot, full vs head-only FT, overfitting diagnostics
summarize_ft_fraction_sensitivity(df_sens, N_REPEATS)

### Fig. A8: RMSE versus fine-tuning fraction

In [ ]:
fig = plot_ft_fraction_sensitivity(
    df_sens,
    "RMSE_annual",
    "RMSE annual (m w.e.)",
    TARGET_REGIONS_SUB,
    fracs=FT_FRACS_SENS,
    strategies=STRATEGIES_SENS,
    main_frac=FT_FRAC,  # dashed line
    exclude_regions={"CENTRALASIA"},
    fname="figures/paperTF/appendix/A8_sensitivity_FT.png")

### Fig. A9: loss curves at 10% fine-tuning data

In [ ]:
fig = plot_loss_grid(
    [(r, FT_FRAC) for r in TARGET_REGIONS_SUB if r != "CENTRALASIA"],
    sens_histories,
    df_sens_sizes,
    strategies=STRATEGIES_SENS,
    n_repeats=N_REPEATS,
    min_ft_seqs=MIN_FT_SEQS,
    row_label_fn=lambda r, f: REGION_LABELS.get(r, r),
    fname="figures/paperTF/appendix/A9_FT_loss_curves.png")